## Requirement to run this code
1- Have HF_Token active on your Colab

2- Put movie.csv in your local Folder

3- Have L4 GPU open in your configuration

The total time of code is around 2 hours


In [1]:
# ==========================================
# 1-a Install/Import package
# ==========================================

!pip install qwen-vl-utils
!pip install -U bitsandbytes>=0.46.1

import requests
import random
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import json
import re
import gc

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
from torch.utils.data import Dataset, DataLoader,TensorDataset
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from sklearn.neural_network import MLPClassifier


## BitsAndBytesConfig

https://huggingface.co/docs/transformers/en/quantization/bitsandbytes

## VLM
https://huggingface.co/docs/transformers/en/tasks/image_text_to_text


In [2]:
# ==========================================
# 1-B Load Vision-Language Model
# ==========================================
model_id = "Qwen/Qwen3-VL-4B-Instruct"
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

model = AutoModelForImageTextToText.from_pretrained(
    model_id, quantization_config=quant_config, device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id, min_pixels=256*28*28, max_pixels=256*28*28)
processor.tokenizer.padding_side = "left"

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

In [3]:
# ==========================================
# 2-A Read-CSV Drop empty url
# ==========================================

df = pd.read_csv('movies.csv', engine='python')
os.makedirs('./posters', exist_ok=True)

# Drop rows where 'Poster_Url' is missing (NaN)
df.dropna(subset=['Poster_Url','Genre'], inplace=True)

# Save the cleaned dataset back to a new CSV (optional but recommended)
df.to_csv('movies_cleaned.csv', index=False)

## ThreadPoolExecutor


https://docs.python.org/3/library/concurrent.futures.html

In [4]:
# ==========================================
# 2-B Download movie posters in multi-threads
# ==========================================

# Add a movie_id column based on the index since it doesn't exist in the CSV
df['movie_id'] = df.index

def download_image(row):
    # Fix column name to match the DataFrame
    url = row['Poster_Url']
    movie_id = row['movie_id']
    save_path = f"./posters/{movie_id}.jpg"

    # Skip if already downloaded
    if os.path.exists(save_path):
        return

    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            with open(save_path, 'wb') as f:
                f.write(response.content)
    except Exception:
        pass # Handle or log errors silently to keep threads moving

# Use ThreadPoolExecutor to download 32 images at a time
records = df.to_dict('records')
with ThreadPoolExecutor(max_workers=32) as executor:
    executor.map(download_image, records)

In [5]:
# ==========================================
# 2-C Sanity Check on the movie poster folder
# ==========================================
poster_dir = './posters'
if os.path.exists(poster_dir):
    downloaded_files = os.listdir(poster_dir)
    num_downloaded = len(downloaded_files)
    total_movies = len(df)

    print(f"Total posters downloaded: {num_downloaded} out of {total_movies}")

    if num_downloaded == total_movies:
        print("All posters have been downloaded successfully!")
    else:
        print(f"Still missing {total_movies - num_downloaded} posters.")
else:
    print("The './posters' directory does not exist yet.")

# Check for missing images
missing_mask = ~df['movie_id'].apply(lambda x: os.path.exists(f"./posters/{x}.jpg"))
missing_count = missing_mask.sum()

print(f"Number of missing pictures: {missing_count}")

# Filter the dataframe if any are missing
if missing_count > 0:
    df = df[~missing_mask].reset_index(drop=True)
    print(f"Dataframe size after removing missing pictures: {len(df)}")
else:
   print("All pictures are present! No need to filter.")

Total posters downloaded: 9826 out of 9826
All posters have been downloaded successfully!
Number of missing pictures: 0
All pictures are present! No need to filter.


# MLB LabelBinarizer

mlb.fit_transform([(1, 2), (3,)])
=> array([[1, 1, 0], [0, 0, 1]])

https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MultiLabelBinarizer.html



In [6]:
# ==========================================
# 2-D Split dataset into 5 equal parts
# ==========================================

# Encode multi-label genres into a binary matrix
mlb = MultiLabelBinarizer()
df['labels'] = mlb.fit_transform(df['Genre'].apply(lambda x: x.split(', '))).tolist()
valid_genres = ", ".join(mlb.classes_)

# Fix random seed for reproducibility
SEED = 50
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Shuffle the dataframe using random seed
df_shuffled = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Split into 5 nearly equal datasets
datasets = np.array_split(df_shuffled, 5)

# Assign to separate variables for easy access later
df_1, df_2, df_3, df_4, df_5 = datasets

# Verify the sizes of each new dataset
for i, d in enumerate(datasets, 1):
    print(f"Dataset {i} size: {len(d)}")


Dataset 1 size: 1966
Dataset 2 size: 1965
Dataset 3 size: 1965
Dataset 4 size: 1965
Dataset 5 size: 1965


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [7]:
# ==========================================
# 3-A Run Zero-Shot on df_1
# ==========================================

# Ensure df_1 has the correct image path pointing to the downloaded posters
df_1 = df_1.copy()
df_1['image_path'] = './posters/' + df_1['movie_id'].astype(str) + '.jpg'

def run_zero_shot(df_subset,prompt, batch_size=32, max_tokens=256):
    predictions = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(df_subset), batch_size), desc="Zero-Shot Inferencing"):
            batch = df_subset.iloc[i:i+batch_size]
            messages = [[{"role": "user", "content": [{"type": "image", "image": row['image_path']}, {"type": "text", "text": prompt}]}] for _, row in batch.iterrows()]

            texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in messages]
            image_inputs, video_inputs = process_vision_info(messages)

            inputs = processor(text=texts, images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(model.device)
            # Use max_tokens parameter here!
            generated_ids = model.generate(**inputs, max_new_tokens=max_tokens)

            trimmed_ids = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
            preds = processor.batch_decode(trimmed_ids, skip_special_tokens=True)
            predictions.extend([p.strip() for p in preds])
    return predictions

prompt_naive = f"""You are an expert movie reviewer. Look at this movie poster.
Based ONLY on the visual style, predict the movie genres.
Choose from the following list: {valid_genres}.
Output ONLY a comma-separated list of genres. Do not explain."""

# Run zero-shot over the df_1 dataset (50 tokens is enough for just genres)
predictions = run_zero_shot(df_1, prompt_naive, max_tokens=50)
df_1['pred_list'] = [[p.strip() for p in pred.split(',') if p.strip() in mlb.classes_] for pred in predictions]

true_matrix = mlb.transform(df_1['Genre'].apply(lambda x: x.split(', ')))
zs_pred_matrix = mlb.transform(df_1['pred_list'])

print(f"\n--- Zero-Shot Results on df_1 ---")
print(f"Exact Match Accuracy: {accuracy_score(true_matrix, zs_pred_matrix):.4f}")
print(f"F1 Score (Micro): {f1_score(true_matrix, zs_pred_matrix, average='micro'):.4f}")

Zero-Shot Inferencing: 100%|██████████| 62/62 [08:22<00:00,  8.11s/it]


--- Zero-Shot Results on df_1 ---
Exact Match Accuracy: 0.0849
F1 Score (Micro): 0.6290


In [8]:
# ==========================================
# 3-B Fine-Tune Prompt
# ==========================================
prompt = f"""You are an expert art director and visual analyst.
Your task is to predict a movie's genres based strictly on the visual composition of its poster.

Step 1: Analyze the visual elements, including the color palette, lighting (e.g., dark/moody vs. bright), typography (e.g., distressed vs. elegant), and iconography/props.
Step 2: Based on this visual evidence, select 1 to 3 genres that best fit.

Rules:
- You must choose ONLY from this exact list: {valid_genres}.
- Ignore your outside knowledge of the movie's plot.
- Ignore the meaning of titles or taglines; evaluate only the aesthetic style of the fonts.

Output your response strictly as a JSON object with the following structure. Do not include markdown formatting or outside text:
{{
  "visual_analysis": "A brief 1-2 sentence description of the visual cues you observed.",
  "genres": ["Genre1", "Genre2"]
}}"""


# Run zero-shot over the df_1 dataset (Give it 300 tokens to finish the JSON!)
predictions = run_zero_shot(df_1, prompt, max_tokens=300)

Zero-Shot Inferencing: 100%|██████████| 62/62 [21:24<00:00, 20.72s/it]


In [9]:
parsed_preds = []
for pred in predictions:
    genres = []
    try:
        # Try strict JSON parsing first
        match = re.search(r'\{.*\}', pred, re.DOTALL)
        if match:
            data = json.loads(match.group(0))
            genres = data.get("genres", [])
    except Exception:
        pass

    # Fallback: If JSON parsing failed, use Regex to salvage the genres array
    if not genres:
        array_match = re.search(r'"genres"\s*:\s*\[(.*?)\]', pred, re.DOTALL | re.IGNORECASE)
        if array_match:
            genres = re.findall(r'"([^"]+)"', array_match.group(1))

    # Ensure it's a list and filter against valid genres
    if isinstance(genres, list):
        parsed_preds.append([g for g in genres if g in mlb.classes_])
    else:
        parsed_preds.append([])

df_1['pred_list'] = parsed_preds

true_matrix = mlb.transform(df_1['Genre'].apply(lambda x: x.split(', ')))
zs_pred_matrix = mlb.transform(df_1['pred_list'])

print(f"\n--- Zero-Shot Results on df_1 (Robust Parsing) ---")
print(f"Exact Match Accuracy: {accuracy_score(true_matrix, zs_pred_matrix):.4f}")
print(f"F1 Score (Micro): {f1_score(true_matrix, zs_pred_matrix, average='micro'):.4f}")


--- Zero-Shot Results on df_1 (Robust Parsing) ---
Exact Match Accuracy: 0.1094
F1 Score (Micro): 0.5999


In [10]:
# Batch 32 is too big for the following part, we use a GC Collector to free RAM
# Run garbage collection to remove unreferenced objects
gc.collect()

# Empty PyTorch's CUDA cache
torch.cuda.empty_cache()

print("CUDA cache cleared!")

CUDA cache cleared!


## Apply_chat_template

https://huggingface.co/docs/transformers/en/chat_templating$0

In [11]:
# ==========================================
# 4-A Feature Extraction (Train: df1,2,3 | Val: df4 | Test: df5)
# ==========================================

# 1. Ensure all subsets have the image_path correctly set
for d in [df_1, df_2, df_3, df_4, df_5]:
    if 'image_path' not in d.columns:
        d['image_path'] = './posters/' + d['movie_id'].astype(str) + '.jpg'

# Note: max_tokens=50 is a generation parameter. For feature extraction,
# we only do a forward pass to get hidden states, so it is not used here.
def extract_embeddings(df_subset, batch_size=16):
    embeddings, labels = [], []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(df_subset), batch_size), desc="Extracting Features"):
            batch = df_subset.iloc[i:i+batch_size]
            # Using prompt to condition the visual extraction
            messages = [[{"role": "user", "content": [{"type": "image", "image": row['image_path']}, {"type": "text", "text": prompt}]}] for _, row in batch.iterrows()]

            texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in messages]
            image_inputs, video_inputs = process_vision_info(messages)
            inputs = processor(text=texts, images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(model.device)

            # Extract the representation of the last token from the last hidden state
            outputs = model(**inputs, output_hidden_states=True)
            embeddings.append(outputs.hidden_states[-1][:, -1, :].cpu().float())
            labels.extend([torch.tensor(row['labels'], dtype=torch.float32) for _, row in batch.iterrows()])

    return torch.cat(embeddings), torch.stack(labels)

# Combine subsets for training
df_train = pd.concat([df_1, df_2, df_3]).reset_index(drop=True)
df_val = df_4.copy()
df_test = df_5.copy()

print("\nExtracting features for Training Set (df1, df2, df3)...")
train_emb, train_lbl = extract_embeddings(df_train)

print("\nExtracting features for Validation Set (df4)...")
val_emb, val_lbl = extract_embeddings(df_val)

print("\nExtracting features for Test Set (df5)...")
test_emb, test_lbl = extract_embeddings(df_test)

print(f"Training embeddings shape: {train_emb.shape}")
print(f"Validation embeddings shape: {val_emb.shape}")
print(f"Test embeddings shape: {test_emb.shape}")



Extracting features for Training Set (df1, df2, df3)...


Extracting Features: 100%|██████████| 369/369 [24:06<00:00,  3.92s/it]



Extracting features for Validation Set (df4)...


Extracting Features: 100%|██████████| 123/123 [08:06<00:00,  3.95s/it]



Extracting features for Test Set (df5)...


Extracting Features: 100%|██████████| 123/123 [08:03<00:00,  3.93s/it]

Training embeddings shape: torch.Size([5896, 2560])
Validation embeddings shape: torch.Size([1965, 2560])
Test embeddings shape: torch.Size([1965, 2560])


In [12]:
# ==========================================
# 4-B Train and Fine-tune Logistic Regression
# ==========================================
print("\n--- Tuning Logistic Regression on df4 (Validation Set) ---")
best_c = 1.0
best_acc = 0.0
best_model = None

# We will test a few different C (inverse of regularization strength) values
C_values = [0.1, 0.3, 1, 3]

for c in C_values:
    # OneVsRest allows LogisticRegression to handle multi-label predictions
    clf = OneVsRestClassifier(LogisticRegression(C=c, max_iter=3000, class_weight='balanced'))
    clf.fit(train_emb.numpy(), train_lbl.numpy())

    # Validate on df4
    val_preds = clf.predict(val_emb.numpy())
    val_acc = accuracy_score(val_lbl.numpy(), val_preds)
    print(f"C={c:5} | Validation Exact Match Accuracy: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        best_c = c
        best_model = clf

print(f"\nBest C parameter found: {best_c} with Validation Exact Match Accuracy: {best_acc:.4f}")

# ==========================================
# 4-C Evaluate on df5 (Test Set)
# ==========================================
print("\n--- Evaluating Best Model on Test Set (df5) ---")
test_preds = best_model.predict(test_emb.numpy())

print(f"Exact Match Accuracy: {accuracy_score(test_lbl.numpy(), test_preds):.4f}")
print(f"F1 Score (Micro): {f1_score(test_lbl.numpy(), test_preds, average='micro'):.4f}")


--- Tuning Logistic Regression on df4 (Validation Set) ---
C=  0.1 | Validation Exact Match Accuracy: 0.0865
C=  0.3 | Validation Exact Match Accuracy: 0.1074
C=    1 | Validation Exact Match Accuracy: 0.1226
C=    3 | Validation Exact Match Accuracy: 0.1288

Best C parameter found: 3 with Validation Exact Match Accuracy: 0.1288

--- Evaluating Best Model on Test Set (df5) ---
Exact Match Accuracy: 0.1221
F1 Score (Micro): 0.6504


In [13]:
# ==========================================
# 4-D Train and Fine-tune MLP
# ==========================================


print("\n--- Tuning MLP on df4 (Validation Set) ---")
best_hidden = (128,)
best_acc = 0.0
best_mlp_model = None

# We will test a few different hidden_layer_sizes
hidden_values = [(128,), (256,), (128, 64)]

for hidden in hidden_values:
    # MLPClassifier supports multi-label classification natively
    clf = MLPClassifier(hidden_layer_sizes=hidden, max_iter=500, random_state=SEED)
    clf.fit(train_emb.numpy(), train_lbl.numpy())

    # Validate on df4
    val_preds = clf.predict(val_emb.numpy())
    val_acc = accuracy_score(val_lbl.numpy(), val_preds)
    print(f"hidden_layer_sizes={hidden} | Validation Exact Match Accuracy: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        best_hidden = hidden
        best_mlp_model = clf

print(f"\nBest hidden_layer_sizes found: {best_hidden} with Validation Exact Match Accuracy: {best_acc:.4f}")

# ==========================================
# 4-E Evaluate MLP on df5 (Test Set)
# ==========================================
print("\n--- Evaluating Best MLP Model on Test Set (df5) ---")
test_preds_mlp = best_mlp_model.predict(test_emb.numpy())

print(f"Exact Match Accuracy: {accuracy_score(test_lbl.numpy(), test_preds_mlp):.4f}")
print(f"F1 Score (Micro): {f1_score(test_lbl.numpy(), test_preds_mlp, average='micro'):.4f}")


--- Tuning MLP on df4 (Validation Set) ---
hidden_layer_sizes=(128,) | Validation Exact Match Accuracy: 0.1496
hidden_layer_sizes=(256,) | Validation Exact Match Accuracy: 0.1567
hidden_layer_sizes=(128, 64) | Validation Exact Match Accuracy: 0.1583

Best hidden_layer_sizes found: (128, 64) with Validation Exact Match Accuracy: 0.1583

--- Evaluating Best MLP Model on Test Set (df5) ---
Exact Match Accuracy: 0.1832
F1 Score (Micro): 0.6210
